In [ ]:
import numpy as np
import pandas as pd

import pyfade

# 1 - Anomaly detection problem

Suppose we have a set of signals, all of them with a sampling period of one hour. These constitute a multidimensional time series. Within these signals, several anomalous behavior occur. We intend to obtain the timestamps where these anomalies appear.

Three methods will be presented here, all based on the calculation of the matrix profile of the signals:

1) Direct calculation of matrix profile on the signal
2) Matrix profile calculation after applying filter
3) Matrix profile on wavelet decomposition

These will be applied in a case study with a signal of increasing complexity.

All matrix profile calculations have these properties:

In [ ]:
SUBSEQ_SIZE = 30
QUANTILE = 0
LEFT_ONLY = False
USE_CUDA = True
SKIP_START=SUBSEQ_SIZE*0

mp_properties ={
    'quantile': QUANTILE,
    'skip_start': SKIP_START,
    'only_left': LEFT_ONLY,
    'use_cuda': USE_CUDA,
}


# 2 - Clean signal with anomalies

## 2.1 - Signal creation

The signals whose anomalies are to be calculated are sinusoidal curves with two types of anomalies:

1) Random noise inserted on given timestamps
2) Inclusion of a unitary step

In [ ]:
# Signal data
num_signals = 5
start_date = '2025-01-01'
end_date = '2025-02-01'
timestamps = pd.date_range(start_date,end_date,freq='h')
amplitudes = [1, 3, 2, 5, 2]
freq = 1/11 # In 1/h


values = np.empty((timestamps.size,num_signals),dtype=float)
time = (timestamps-timestamps[0]).total_seconds()/3600
for s in range(num_signals):
    values[:,s] = amplitudes[s] * np.sin(2*freq*time)

orig_signals = pd.DataFrame(values,index=timestamps)
orig_signals.columns = [f'Signal {k}' for k in range(num_signals)]
orig_signals.index = pd.to_datetime(orig_signals.index)

pyfade.plot_multiple(orig_signals);


Now to include the anomalies. This is done in two steps, first we include the random anomalies, then, the step ones.

In [ ]:
# Size of the anomalies in hours
size = 24

# Setting up seed so anomalies are always the same when all inputs are the same
rng = np.random.default_rng(2025)

# Anomaly location (# signals with anomaly, start, size)
rand_anomalies = [(2, '2025-01-10', size, []),
                  (4, '2025-01-20', size, []),
                  (1, '2025-01-25', size, [])]

step_anomalies = [(2, '2025-01-13', size, []),
                  (1, '2025-01-28', size, [])]

# Inserting random anomalies
anom_signal = orig_signals.copy()
for num_anomalous,start,size,an_list in rand_anomalies:

    if num_anomalous > num_signals:
        num_anomalous = num_signals

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, periods=size, freq='h'),col] += 1 *amplitudes[s] * rng.random(size)

# Inserting step anomalies
for num_anomalous,start,_,an_list in step_anomalies:

    if num_anomalous >= num_signals:
        num_anomalous = num_signals

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, end=anom_signal.index.max(), freq='h'),col] += 1 *amplitudes[s]

pyfade.plot_multiple(anom_signal);

In [ ]:
# Defining function to plot anomalies
def plot_anomalies(axs,subseq_size):
    for _,start,size,series in rand_anomalies:
        start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
        for s in series:
            axs[s].axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

    for _,start,size,series in step_anomalies:
        start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
        for s in series:
            axs[s].axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

def plot_anomalies_kdp(axs,subseq_size):

    for k in range(num_signals):
        for num_anomalous,start,size,_ in rand_anomalies:
            start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
            if k < num_anomalous:
                axs[k].axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

        for num_anomalous,start,size,_ in step_anomalies:
            start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
            if k < num_anomalous:
                axs[k].axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

def plot_wavelet(axs,subseq_size,signal):

    for _,start,size,series in rand_anomalies:
        start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
        if signal in series:
            for ax in axs:
                ax.axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

    for _,start,size,series in step_anomalies:
        start = pd.Timestamp(start)-pd.Timedelta(hours=subseq_size)
        if signal in series:
            for ax in axs:
                ax.axvspan(start,start+pd.Timedelta(hours=size+subseq_size),color='gray',alpha=0.5)

## 2.2 - Directly on the signal

### 2.2.1 - Matrix Profile

This is a pretty simple signal, we see that just calculating its matrix profile is enough to identify all the anomalies without any issues.

In [ ]:
from matplotlib import pyplot as plt
MP = pyfade.get_MP(anom_signal,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

### 2.2.2 - K-Dimensional Profile

In [ ]:
KDP = pyfade.get_KDP(anom_signal,SUBSEQ_SIZE,pre_calc_MP=MP,only_left=LEFT_ONLY,skip_start=SKIP_START)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

## 2.3 - Filtered Matrix Profile

Now, a Butterworth pass band filter will be used. The lower and upper frequencies are given as periods.

### 2.3.1 - Butterworth Band: 3rd order 24h - 4h

In [ ]:
T_cutoff = np.array([24, 4])
freq_cut = 1/T_cutoff
order = 3

num_coef, den_coef =pyfade.get_filter(pyfade.Filter_type.BUTTER_PASS, cutoff_freq=freq_cut, order=order)
signal_filtered = pyfade.apply_filter(anom_signal, num_coef, den_coef)
fig, axs = pyfade.plot_multiple(signal_filtered, ylabel='Filtered')
axs[0].set_title('Filtered signals');

In [ ]:
MP = pyfade.get_MP(signal_filtered,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(signal_filtered,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

Once again, not much difference is seen here, as the signal is well-behaved. Since the period of the original signal is 11 h, note what happens when we exclude it from the band-pass interval:

In [ ]:
T_cutoff = np.array([30, 18])
freq_cut = 1/T_cutoff
order = 3

num_coef, den_coef =pyfade.get_filter(pyfade.Filter_type.BUTTER_PASS, cutoff_freq=freq_cut, order=order)
signal_filtered = pyfade.apply_filter(anom_signal, num_coef, den_coef)
fig, axs = pyfade.plot_multiple(signal_filtered, ylabel='Filtered')
axs[0].set_title('Filtered signals');

In [ ]:
MP = pyfade.get_MP(signal_filtered,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(signal_filtered,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

As we removed the periodic regular behavior, the algorithm struggles to find the motifs and discords. Although the matrix profile values do indeed increase where we expect, it also peaks where it shouldn't. Thus, we must make sure we do not filter out regular healthy components from our signal before filtering.

## 2.4 - Wavelet Matrix Profile

On this last approach, we decompose the signal using a multi-level wavelet decomposition. For instance, let us check the decomposition of the first signal with a level of 3:

In [ ]:
wavelet_name = 'db2'
decomp_level = 3

signals,_ = pyfade.get_signal_decomp(anom_signal.iloc[:,0],wavelet=wavelet_name,create_plot=True,level=decomp_level);

The curve above shows the original signal and each decomposition level. Each curve shows the reconstructed signal considering only the coefficients from that term. That is, the curves to the left show the reconstruction from the approximation coefficients, while the ones to the right, the reconstruction from the detail coefficients.

For this signal, we can calculate the matrix profile of each successive detail reconstructed curve and on the final approximation one:

In [ ]:
MP,_ = pyfade.get_MP_from_wavelets(anom_signal.iloc[:,0],SUBSEQ_SIZE,level=decomp_level,wavelet=wavelet_name,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
axs[0].set_title('Matrix Profile');

In [ ]:
MP = pyfade.wavelet_MP_from_KDP(anom_signal,SUBSEQ_SIZE,1,level=decomp_level,wavelet=wavelet_name,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(anom_signal,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

# 3 - Noisy signal

In [ ]:
# Properties
LEVEL_OF_NOISE = 1

## 3.1 - Signal creation

In [ ]:
# Signal data
num_signals = 5
start_date = '2025-01-01'
end_date = '2025-02-01'
timestamps = pd.date_range(start_date,end_date,freq='h')
amplitudes = [1, 3, 2, 5, 2]
freq = 1/11 # In 1/h

# Setting up seed so randomness is always the same with same given signal
rng = np.random.default_rng(2025)

values = np.empty((timestamps.size,num_signals),dtype=float)
time = (timestamps-timestamps[0]).total_seconds()/3600
for s in range(num_signals):
    values[:,s] = amplitudes[s] * np.sin(2*freq*time)
    values[:,s] += LEVEL_OF_NOISE *amplitudes[s] * rng.random(time.size)


orig_signals = pd.DataFrame(values,index=timestamps)
orig_signals.columns = [f'Signal {k}' for k in range(num_signals)]
orig_signals.index = pd.to_datetime(orig_signals.index)

pyfade.plot_multiple(orig_signals);


In [ ]:
# Size of the anomalies in hours
size = 24

# Setting up seed so anomalies are always the same when all inputs are the same
rng = np.random.default_rng(2025)

# Anomaly location (# signals with anomaly, start, size)
rand_anomalies = [(2, '2025-01-10', size, []),
                  (4, '2025-01-20', size, []),
                  (1, '2025-01-25', size, [])]

step_anomalies = [(2, '2025-01-13', size, []),
                  (1, '2025-01-28', size, [])]

# Inserting random anomalies
anom_signal = orig_signals.copy()
for num_anomalous,start,size,an_list in rand_anomalies:

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, periods=size, freq='h'),col] += 1 *amplitudes[s] * rng.random(size)

# Inserting step anomalies
for num_anomalous,start,_,an_list in step_anomalies:

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, end=anom_signal.index.max(), freq='h'),col] += 1 *amplitudes[s]

pyfade.plot_multiple(anom_signal);

## 3.2 - Directly on signal

### 3.2.1 - Matrix Profile

In [ ]:
MP = pyfade.get_MP(anom_signal,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

### 3.2.2 - K-Dimensional Profile

In [ ]:
KDP = pyfade.get_KDP(anom_signal,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

## 3.3 - Filtered Matrix Profile

In [ ]:
T_cutoff = np.array([24, 6])
freq_cut = 1/T_cutoff
order = 3

num_coef, den_coef =pyfade.get_filter(pyfade.Filter_type.BUTTER_PASS, cutoff_freq=freq_cut, order=order)
signal_filtered = pyfade.apply_filter(anom_signal, num_coef, den_coef)
fig, axs = pyfade.plot_multiple(signal_filtered, ylabel='Filtered')
axs[0].set_title('Filtered signals');

In [ ]:
MP = pyfade.get_MP(signal_filtered,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(signal_filtered,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

## 3.4 - Wavelet Matrix Profile

In [ ]:
wavelet_name = 'db2'
decomp_level = 4
analyzed_signal = 3

In [ ]:
MP,_ = pyfade.get_MP_from_wavelets(anom_signal.iloc[:,analyzed_signal],SUBSEQ_SIZE,level=decomp_level,wavelet=wavelet_name,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
axs[0].set_title('Matrix Profile');
plot_wavelet(axs,SUBSEQ_SIZE,analyzed_signal)

In [ ]:
MP = pyfade.wavelet_MP_from_KDP(anom_signal,SUBSEQ_SIZE,analyzed_signal,level=decomp_level,wavelet=wavelet_name,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(anom_signal,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

# 4 - Signal with a lot of frequency components

In [ ]:
# Properties
PERIODS_AND_AMPLITUDES = [
    (5, 1),
    (15, 0.5),
    (20, 2),
    (23, 0.7),
    (11, 0.3),
    (1, 1000)
]

## 4.1 - Signal creation

In [ ]:
# Signal data
order = np.random.default_rng(2026)
num_signals = 5
start_date = '2025-01-01'
end_date = '2025-02-01'
timestamps = pd.date_range(start_date,end_date,freq='h')
amplitudes = [1* order.random(1), 3* order.random(1), 2* order.random(1), 5* order.random(1), 2* order.random(1)]
freq = 1/11 # In 1/h




values = np.empty((timestamps.size,num_signals),dtype=float)
time = (timestamps-timestamps[0]).total_seconds()/3600
for s in range(num_signals):
    values[:,s] = amplitudes[s] * np.sin(2*freq*time)
    for T, A in PERIODS_AND_AMPLITUDES:
        values[:,s] += A * amplitudes[s] * np.sin(2*(1/T)*time)


orig_signals = pd.DataFrame(values,index=timestamps)
orig_signals.columns = [f'Signal {k}' for k in range(num_signals)]
orig_signals.index = pd.to_datetime(orig_signals.index)

pyfade.plot_multiple(orig_signals);


In [ ]:
# Size of the anomalies in hours
size = 24

# Setting up seed so anomalies are always the same when all inputs are the same
rng = np.random.default_rng(2025)

# Anomaly location (# signals with anomaly, start, size)
rand_anomalies = [(2, '2025-01-10', size, []),
                  (4, '2025-01-20', size, []),
                  (1, '2025-01-25', size, [])]

step_anomalies = [(2, '2025-01-13', size, []),
                  (1, '2025-01-28', size, [])]

# Inserting random anomalies
anom_signal = orig_signals.copy()

Amp = np.sqrt(np.sum(np.square([a[1] for a in PERIODS_AND_AMPLITUDES])))
for num_anomalous,start,size,an_list in rand_anomalies:

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, periods=size, freq='h'),col] += 1 * (amplitudes[s]+Amp) * rng.random(size)*rng.random(1)

# Inserting step anomalies
for num_anomalous,start,_,an_list in step_anomalies:

    # Getting information, choosing faulty signals and saving this info
    series = rng.choice(num_signals,size=num_anomalous,replace=False)
    an_list.extend(series)

    for s in series:
        col = anom_signal.columns[s]
        anom_signal.loc[pd.date_range(start=start, end=anom_signal.index.max(), freq='h'),col] += 1 *amplitudes[s]

pyfade.plot_multiple(anom_signal);

## 4.2 - Directly on signal

In [ ]:
MP = pyfade.get_MP(anom_signal,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

In [ ]:
KDP = pyfade.get_KDP(anom_signal,SUBSEQ_SIZE,pre_calc_MP=MP)
fig, axs = pyfade.plot_multiple(KDP);
plot_anomalies_kdp(axs,SUBSEQ_SIZE)
axs[0].set_title('K-Dimensional Profile');

## 4.3 - Filtered Matrix Profile

In [ ]:
T_cutoff = np.array([24, 4])
freq_cut = 1/T_cutoff
order = 3

num_coef, den_coef =pyfade.get_filter(pyfade.Filter_type.BUTTER_PASS, cutoff_freq=freq_cut, order=order)
signal_filtered = pyfade.apply_filter(anom_signal, num_coef, den_coef)
fig, axs = pyfade.plot_multiple(signal_filtered, ylabel='Filtered')
axs[0].set_title('Filtered signals');

In [ ]:
MP = pyfade.get_MP(signal_filtered,SUBSEQ_SIZE,**mp_properties)
fig, axs = pyfade.plot_multiple(MP);
plot_anomalies(axs,SUBSEQ_SIZE)
axs[0].set_title('Matrix Profile');

## A

# 6 - Signal with frequency components and noise